# Excitation Exchange Coupling Between Chromophores

This notebook computes the **third-order nonlinear (2D) spectroscopic response** of two coupled two-level chromophores (A and B) in the impulsive regime, using `qudpy` for the quantum dynamics and `ufss` for the double-sided Feynman diagram generation.

The system consists of:
- Two chromophores with energies $E_1 = 2\,\text{eV}$ and $E_2 = 2.1\,\text{eV}$
- A dipole–dipole (excitation exchange) coupling $J = 0.3\,\text{eV}$
- A thermal bath at $T = 300\,\text{K}$ modelled via Lindblad collapse operators

We compute both the **rephasing** ($R_1, R_2, R_3$) and **non-rephasing** ($R_4, R_5, R_6$) pathways and plot the resulting 2D spectra.

## 1. Imports

In [ ]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt

from qudpy.Classes import *
import qudpy.plot_functions as pf
import ufss  # diagram generation

## 2. Generate Double-Sided Feynman Diagrams

We use `ufss.DiagramGenerator` to produce the six third-order diagrams in the impulsive regime.

- **Rephasing** (phase-matching condition $-\mathbf{k}_1 + \mathbf{k}_2 + \mathbf{k}_3$): diagrams $R_1, R_2, R_3$
- **Non-rephasing** (phase-matching condition $+\mathbf{k}_1 - \mathbf{k}_2 + \mathbf{k}_3$): diagrams $R_4, R_5, R_6$

In [ ]:
DG = ufss.DiagramGenerator
R3rd = DG()

# Pulse envelope (impulsive limit: very short pulse)
t_pulse = np.array([-1, 1])
R3rd.efield_times = [t_pulse] * 4

# --- Rephasing diagrams ---
# Phase discrimination: (-k1, +k2, +k3)
R3rd.set_phase_discrimination([(0, 1), (1, 0), (1, 0)])
[R3, R1, R2] = R3rd.get_diagrams([0, 100, 200, 300])
rephasing = [R1, R2, R3]
print('Rephasing diagrams (R1, R2, R3):')
for name, d in zip(['R1', 'R2', 'R3'], rephasing):
    print(f'  {name}: {d}')

# --- Non-rephasing diagrams ---
# Phase discrimination: (+k1, -k2, +k3)
R3rd.set_phase_discrimination([(1, 0), (0, 1), (1, 0)])
[R6, R4, R5] = R3rd.get_diagrams([0, 100, 200, 200])
nonrephasing = [R4, R5, R6]
print('\nNon-rephasing diagrams (R4, R5, R6):')
for name, d in zip(['R4', 'R5', 'R6'], nonrephasing):
    print(f'  {name}: {d}')

## 3. Build the Hamiltonian

The system Hamiltonian is:

$$H = \hbar\omega_1 a^\dagger a + \hbar\omega_2 b^\dagger b + \hbar J(a^\dagger b + b^\dagger a)$$

where $a$, $b$ are the lowering operators for chromophores A and B respectively, and $J$ is the excitation exchange coupling.

Both chromophores are modelled as **two-level systems** ($|g\rangle$, $|e\rangle$), giving a 4-dimensional Hilbert space: $\{|gg\rangle, |eg\rangle, |ge\rangle, |ee\rangle\}$.

In [ ]:
hbar = 0.658211951  # eV·fs

# Site energies
E1 = 2.0   # eV  (chromophore A)
E2 = 2.1   # eV  (chromophore B)
w1 = E1 / hbar
w2 = E2 / hbar

# Coupling
j = 0.3 / hbar  # excitation exchange coupling (rad/fs)

# Dipole strengths (equal here)
mu1 = 1.0
mu2 = 1.0

# Operators in the composite Hilbert space (A ⊗ B)
a  = tensor(destroy(2), qeye(2))    # lowering operator: chromophore A
b  = tensor(qeye(2),   destroy(2))  # lowering operator: chromophore B
A  = mu1*a + mu2*b                  # total lowering operator
mu = mu1*(a.dag()+a) + mu2*(b.dag()+b)  # total dipole operator

# Hamiltonian
H  = hbar * (w1*a.dag()*a + w2*b.dag()*b)  # free evolution
H += hbar * j * (a.dag()*b + b.dag()*a)    # coupling term

print('Hamiltonian (in eV):')
print(H * hbar)  # display in eV by reversing the hbar factor

## 4. Collapse Operators (System–Bath Coupling)

We model the thermal bath via **Lindblad collapse operators**, including both relaxation and thermally-driven excitation at temperature $T = 300\,\text{K}$:

$$c_\text{relax} = \sqrt{\kappa(\bar{n}+1)}\,a, \qquad c_\text{excite} = \sqrt{\kappa\,\bar{n}}\,a^\dagger$$

where $\bar{n} = (e^{E/k_BT}-1)^{-1}$ is the thermal occupation number.

In [ ]:
kappa = 0.1                        # system-bath coupling strength (eV)
kB    = 8.617333262e-5             # Boltzmann constant (eV/K)
T     = 300                        # temperature (K)
beta  = 1 / (T * kB)

# Thermal occupation numbers for each uncoupled site
n1 = 1 / (np.exp(E1 * beta) - 1)
n2 = 1 / (np.exp(E2 * beta) - 1)
print(f'Thermal occupations:  n1 = {n1:.4f},  n2 = {n2:.4f}')

# Collapse operators
c1 = np.sqrt(kappa * (n1 + 1)) * a    # relaxation: A
c2 = np.sqrt(kappa * (n2 + 1)) * b    # relaxation: B
c3 = np.sqrt(kappa * n1) * a.dag()    # excitation:  A
c4 = np.sqrt(kappa * n2) * b.dag()    # excitation:  B
c_ops = [c1, c2, c3, c4]

## 5. Initialise the `System` Object

The initial state is the global ground state $\rho_0 = |gg\rangle\langle gg|$. We pass `diagonalize=True` so that all operators are automatically transformed into the energy eigenbasis of $H$ before the simulation — this is important for accurately capturing the exciton (eigenstates) picture of the coupled dimer.

In [ ]:
rho0 = tensor(fock_dm(2, 0), fock_dm(2, 0))  # |gg><gg|

sys = System(
    H      = H,
    rho    = rho0,
    a      = A,
    u      = mu,
    c_ops  = c_ops,
    diagonalize = True   # transform into exciton (eigen-energy) basis
)

# Eigenvalues for reference
en, evecs = H.eigenstates()
print('\nEigenvalues of H (eV·rad/fs):', en)
print('Eigenvalues of H (eV):', en * hbar)

## 6. Compute the 2D Coherence Response

We compute the third-order dipole response for each of the 6 diagrams, scanning over two coherence time delays ($t_1$ and $t_3$) with a fixed population time. The response is multiplied by $i$ to convert from the rotating frame convention used internally.

In [ ]:
# Time delays: [t1 (coherence), t2 (population), t3 (coherence)]
# scan_id marks which delays are scanned (t1=index 0, t3=index 2)
time_delays = [80, 20, 80]  # fs
scan_id = [0, 2]

total_diagrams = rephasing + nonrephasing
response_list  = []

for k, diagram in enumerate(total_diagrams):
    label = ['R1','R2','R3','R4','R5','R6'][k]
    print(f'Computing {label}...')
    t1, t2, dipole = sys.coherence2d(
        time_delays, diagram, scan_id,
        r=1, parallel=False  # note: parallel not supported in this version
    )
    response_list.append(1j * dipole)

print('\nAll diagrams computed.')

## 7. Fourier Transform to Frequency Domain

We apply a 2D Fourier transform to the time-domain dipole responses to obtain the 2D spectra in the $(\omega_1, \omega_3)$ frequency domain.

In [ ]:
spectra_list, extent, f1, f2 = sys.spectra(
    np.imag(response_list),
    resolution=1
)

print(f'Frequency axis range: {extent[0]:.2f} to {extent[1]:.2f} eV')
print(f'Number of spectra computed: {len(spectra_list)}')

## 8. Plot Rephasing 2D Spectra

Rephasing pathways produce the photon-echo signal. The diagonal peaks correspond to transitions at $E_1$ and $E_2$, while the cross-peaks arise from the excitation exchange coupling $J$.

In [ ]:
rephasing_spectra = spectra_list[:3]
rephasing_spectra.append(np.sum(spectra_list[:3], axis=0))  # total rephasing

pf.plot_contourf_multi_spectra_norm(
    rephasing_spectra, f1, f2,
    labels     = ['E emission (eV)', 'E absorption (eV)'],
    title_list = ['$R_1$', '$R_2$', '$R_3$', '$R_{\\mathrm{rephasing}}$'],
    scale      = 'linear',
    color_map  = 'jet',
    center_scale = False,
    plot_sum   = False,
    plot_quadrant = 'All',
    invert_y   = False,
    diagonals  = [True, False],
    nlevels    = 30,
    Zoom_coor  = [2, 3, -3, -2]
)

## 9. Plot Non-Rephasing 2D Spectra

Non-rephasing pathways do not produce a photon echo; when added to the rephasing signal they give the absorptive (purely real) 2D spectrum. Cross-peak asymmetry between rephasing and non-rephasing can be used to determine the direction of energy transfer.

In [ ]:
nonrephasing_spectra = spectra_list[3:]
nonrephasing_spectra.append(np.sum(spectra_list[3:], axis=0))  # total non-rephasing

pf.plot_contourf_multi_spectra_norm(
    nonrephasing_spectra, f1, f2,
    labels     = ['E emission (eV)', 'E absorption (eV)'],
    title_list = ['$R_4$', '$R_5$', '$R_6$', '$R_{\\mathrm{nonrephasing}}$'],
    scale      = 'linear',
    color_map  = 'jet',
    center_scale = False,
    plot_sum   = False,
    plot_quadrant = 'All',
    invert_y   = False,
    diagonals  = [False, True],
    nlevels    = 30,
    Zoom_coor  = [2, 3, 2, 3]
)

## Notes

- The `parallel=True` option in `coherence2d` is **not active** in this version of qudpy — computations run sequentially. For large time grids, consider using `coherence2d_s` (the optimised sequential variant).
- The `spectra()` method uses `np.fft.ifft2` (inverse FFT convention). Verify that peak positions match your expected sign convention for rephasing/non-rephasing pathways.
- Cross-peak amplitudes scale with the coupling $J$: setting `j = 0` should suppress them and recover two independent two-level system responses.
- The population time $t_2 = 20\,\text{fs}$ here. Repeating the calculation for different values of `time_delays[1]` produces a population-time series (equivalent to calling `sys.pop_study`).